# Cleaning and feature engineering

All the cleaning lives in src/features.py so the same code runs on train, on every CV fold and on test.

Two kinds of steps:
- clean() works row by row and learns nothing from the data, so it's safe to run anywhere.
- Anything that learns from data (medians, which columns are skewed, scaling, encoding) lives inside the Pipeline, so in CV it only ever sees the training fold.

In [1]:
# Make src/ importable from notebooks/ and reload edited modules automatically
%load_ext autoreload
%autoreload 2
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd

from src.cv import RAW_DIR
from src.features import clean

Same setup as the EDA notebook. clean() comes from src/features.py, so the exact same cleaning runs on train, on every CV fold and on test.

In [2]:
train = pd.read_csv(RAW_DIR / "train.csv")
test = pd.read_csv(RAW_DIR / "test.csv")

# Features only: Id is just a row label, SalePrice is the target
X_train = clean(train.drop(columns=["Id", "SalePrice"]))
X_test = clean(test.drop(columns=["Id"]))
print(X_train.shape, X_test.shape)

# After cleaning, only TRULY missing values should remain
remaining = pd.DataFrame({"train": X_train.isna().sum(), "test": X_test.isna().sum()})
print(remaining[remaining.sum(axis=1) > 0])

(1460, 91) (1459, 91)
              train  test
MSZoning          0     4
LotFrontage     259   227
Utilities         0     2
Exterior1st       0     1
Exterior2nd       0     1
MasVnrType        5     3
BsmtQual          0     2
BsmtCond          0     3
BsmtExposure      1     2
BsmtFinType2      1     0
Electrical        1     0
KitchenQual       0     1
Functional        0     2
GarageFinish      0     2
GarageCars        0     1
GarageArea        0     1
GarageQual        0     2
GarageCond        0     2
PoolQC            0     3
SaleType          0     1


91 columns: the 79 original ones plus 5 has_ flags and 7 new features. Train and test end up with the same columns.

What's still missing is only the truly missing data: LotFrontage (259 in train, 227 in test), a few single values in train (Electrical, BsmtExposure, BsmtFinType2, MasVnrType), and in test some houses that have a pool, basement or garage but no recorded quality. The pipeline will impute these.

The 81 no-garage and 37 no-basement houses are no longer in the list, so the "absent" filling worked. GarageYrBlt isn't there either because it was filled with YearBuilt (including the 2207 typo in test).

In [3]:
# Sanity check of the new features and flags
new_cols = ["TotalSF", "TotalBath", "HouseAge", "RemodAge", "is_remodeled", "TotalPorchSF", "QualxArea"]
print(X_train[new_cols].describe().T.round(1))

flags = ["has_basement", "has_garage", "has_pool", "has_fireplace", "has_2nd_floor"]
print(X_train[flags].mean().round(3))

# Do the new features beat their parts? Correlation with log price
y = np.log1p(train["SalePrice"])
print(X_train[new_cols + ["GrLivArea", "TotalBsmtSF", "FullBath"]].corrwith(y).sort_values(ascending=False).round(3))

               count    mean     std    min     25%     50%      75%      max
TotalSF       1460.0  2567.0   821.7  334.0  2009.5  2474.0   3004.0  11752.0
TotalBath     1460.0     2.2     0.8    1.0     2.0     2.0      2.5      6.0
HouseAge      1460.0    36.5    30.3    0.0     8.0    35.0     54.0    136.0
RemodAge      1460.0    23.0    20.6    0.0     4.0    14.0     41.0     60.0
is_remodeled  1460.0     0.4     0.5    0.0     0.0     0.0      1.0      1.0
TotalPorchSF  1460.0    87.1   105.2    0.0     0.0    48.0    136.0   1027.0
QualxArea     1460.0  9674.0  5186.7  334.0  5790.0  8820.0  12180.0  56420.0
has_basement     0.975
has_garage       0.945
has_pool         0.005
has_fireplace    0.527
has_2nd_floor    0.432
dtype: float64
QualxArea       0.800
TotalSF         0.777
GrLivArea       0.701
TotalBath       0.673
TotalBsmtSF     0.612
FullBath        0.595
TotalPorchSF    0.195
is_remodeled    0.164
RemodAge       -0.568
HouseAge       -0.587
dtype: float64


The flags match the EDA: 94.5% of houses have a garage, 97.5% a basement, 52.7% a fireplace. Only 7 houses have a pool, so has_pool won't help much.

The max of TotalSF (11,752) and QualxArea (56,420) is house 1299, the big outlier from the EDA again.

The new features correlate better than their parts: QualxArea 0.80 vs GrLivArea 0.70, TotalSF 0.78 vs TotalBsmtSF 0.61, TotalBath 0.67 vs FullBath 0.60. This is still one feature at a time, though; the models will show whether they add anything on top of the originals.

HouseAge (-0.59) is basically YearBuilt flipped, because YrSold only covers 5 years. That's another redundant pair for the linear models.

is_remodeled was -0.07 at first, which didn't make sense. YearRemodAdd never goes below 1950, so 166 old houses that were never remodeled showed 1950 and were counted as remodeled. After excluding them, the correlation became +0.16: remodeled houses sell a bit higher. It's still small, partly because remodeled houses also tend to be older.

In [4]:
# YearRemodAdd starts at 1950: how many old houses sit exactly on that floor?
print("min YearRemodAdd:", train["YearRemodAdd"].min())
print("built before 1950 with remodel year 1950:",
      ((train["YearRemodAdd"] == 1950) & (train["YearBuilt"] < 1950)).sum())

min YearRemodAdd: 1950
built before 1950 with remodel year 1950: 166


166 houses built before 1950 show a remodel year of exactly 1950. That's the floor of the column, not real remodels, so clean() no longer counts them as remodeled.

## The pipeline

The pipeline goes from raw data to model input. fit() learns what each step needs from the training data only; transform() applies it to new data without learning anything. Inside cross_val_score this happens again for every fold, so the validation fold never affects any median or scaler.

In [5]:
from src.features import make_linear_preprocessor, make_tree_preprocessor

X = train.drop(columns=["Id", "SalePrice"])
X_test_raw = test.drop(columns=["Id"])

# Fit on train only, then apply the same learned steps to test
prep = make_linear_preprocessor()
Xt_train = prep.fit_transform(X)
Xt_test = prep.transform(X_test_raw)
print("linear:", Xt_train.shape, Xt_test.shape, "| NaNs:", np.isnan(Xt_train).sum(), np.isnan(Xt_test).sum())

tree_prep = make_tree_preprocessor()
Tt_train = tree_prep.fit_transform(X)
print("tree:  ", Tt_train.shape, "| NaNs:", np.isnan(Tt_train).sum())

linear: (1460, 280) (1459, 280) | NaNs: 0 0
tree:   (1460, 91) | NaNs: 0


The linear version has 280 columns because one-hot turns each text column into several 0/1 columns. Train and test get exactly the same 280: the categories are learned from train, and anything new in test is ignored instead of creating a new column. The tree version keeps 91 columns, with categories as integer codes. No NaNs left in either, so all the truly missing values got imputed.

In [6]:
# Which numeric columns did the pipeline decide to log-transform?
cols = prep.named_steps["columns"]
num_names = [c.removeprefix("num__") for c in cols.get_feature_names_out() if c.startswith("num__")]
logged = cols.named_transformers_["num"].named_steps["log_skewed"].columns_
print(len(logged), "log-transformed:", [num_names[i] for i in logged])

# LotFrontage medians learned from the data it was fitted on
lf = prep.named_steps["lot_frontage"]
print(lf.group_medians_.sort_values().round(1).to_dict())
print("global fallback:", lf.global_median_)

20 log-transformed: ['LotFrontage', 'LotArea', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'MiscVal', 'TotalSF', 'TotalPorchSF', 'QualxArea']
{'BrDale': 21.0, 'MeadowV': 21.0, 'NPkVill': 24.0, 'Blueste': 24.0, 'Blmngtn': 43.0, 'BrkSide': 52.0, 'IDOTRR': 60.0, 'OldTown': 60.0, 'SWISU': 60.0, 'StoneBr': 61.5, 'Gilbert': 65.0, 'Edwards': 65.5, 'SawyerW': 66.5, 'Veenker': 68.0, 'CollgCr': 70.0, 'Sawyer': 71.0, 'NAmes': 73.0, 'Mitchel': 73.0, 'Somerst': 73.5, 'Crawfor': 74.0, 'ClearCr': 80.0, 'NWAmes': 80.0, 'Timber': 85.0, 'NridgHt': 88.5, 'NoRidge': 91.0}
global fallback: 69.0


20 columns get log-transformed, all continuous measurements: areas, LotFrontage, MiscVal and the engineered area features. At first the rule also picked ordinal scales (ExterQual, PoolQC...), has_pool and small counts like KitchenAbvGr. They looked skewed only because most houses share one value, and logging them would make the steps of a quality scale uneven. Now the log only applies to columns with more than 10 distinct values.

The LotFrontage medians match the EDA, from 21 ft (BrDale, MeadowV) to 91 ft (NoRidge), with 69 as the global fallback. Here they’re fitted on the full train set just to look at them; inside CV they’re learned again on each training fold.

In [7]:
# Smoke test only: does the full pipeline run inside CV without errors?
# Not logged to results.csv; the real validation protocol comes in Phase 3.
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from src.cv import SEED

y = np.log1p(train["SalePrice"])
model = make_pipeline(make_linear_preprocessor(), Ridge(alpha=10))
scores = -cross_val_score(model, X, y, cv=KFold(5, shuffle=True, random_state=SEED),
                          scoring="neg_root_mean_squared_error")
print(scores.round(4), "mean", scores.mean().round(4))

[0.1278 0.1231 0.178  0.1193 0.1096] mean 0.1315


The full pipeline runs inside 5-fold CV with Ridge, with a mean RMSE of about 0.13 on log price. Not logged yet; this only checks that everything works fold by fold. Fold 3 is much worse than the others (0.178 vs about 0.12).

Restricting the log to continuous columns moved the mean from 0.1328 to 0.1315, with 4 of 5 folds a little better. That’s tiny compared to the spread between folds, so the fix is right in principle but not a real gain.

In [8]:
# Which fold holds the 4 very large houses?
cv = KFold(5, shuffle=True, random_state=SEED)
for k, (tr_idx, va_idx) in enumerate(cv.split(X)):
    big_in_fold = sorted(set(train.loc[va_idx, "Id"]) & {524, 692, 1183, 1299})
    print(f"fold {k + 1}: {big_in_fold}")

fold 1: [692]
fold 2: [1183]
fold 3: [524, 1299]
fold 4: []
fold 5: []


Fold 3 is the only fold with both 524 and 1299, the two huge Edwards houses that sold cheap. That’s why its RMSE is 0.18 while the other folds are around 0.12. The model predicts a mansion price for them (roughly 5 times what they sold for), and since errors are squared, those two houses alone push the whole fold up. Folds 1 and 2 each have one of the big NoRidge houses, but those follow the trend, so the folds score normally. The outlier experiment in Phase 5 will test what happens without 524 and 1299.